In [0]:
pip install gtfs-kit

In [0]:
pip install gtfs-realtime-bindings

In [0]:
%restart_python

In [0]:
# Testing secret scope for the Translink API key

SCOPE_NAME = 'GTFS-INTERNAL' 
SECRET_KEY = 'TranslinkAPIKey'

print("Listing secrets in scope:", dbutils.secrets.list(scope=SCOPE_NAME))

# The GET will succeed and return [REDACTED]
API_KEY = dbutils.secrets.get(scope=SCOPE_NAME, key=SECRET_KEY)
print(f"Key retrieval successful: {API_KEY}")

In [0]:
import requests
from google.transit import gtfs_realtime_pb2 # Required for parsing GTFS-RT Protobuf data

import time
import os

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType


API_KEY = dbutils.secrets.get(scope="GTFS-INTERNAL", key="TranslinkAPIKey")
API_URL = "https://gtfsapi.translink.ca/v3/gtfsposition"

# Define the Bronze Landing Zone for the raw real-time stream
REALTIME_BRONZE_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/realtime_bronze/vehicle_positions"

In [0]:
#Fetches GTFS-RT Vehicle Positions and saves the Protobuf content to a file

def fetch_and_save_realtime_data(url: str, api_key: str, output_path: str):
    full_url = f"{url}?apikey={api_key}"
    # telling servers to send data in GTFS-RT (x)protobuf format
    # and authenticate with translink API
    headers = {
        'Accept': 'application/x-protobuf'
    }
    
    try:
        response = requests.get(full_url, headers=headers, timeout=10)
        response.raise_for_status() 
        
        # 1. Parse the Protocol Buffer data
        feed = gtfs_realtime_pb2.FeedMessage() # feedMessage: object that represents a GTFS feed
        feed.ParseFromString(response.content)  # from the API response, convert binary to Python object
        
        # 2. Extract key fields into a list of dictionaries (micro-batch)
        data_list = []
        timestamp = time.time() # Capture the ingestion time

        # feed.entity contains a list of vehicles, trips, or alerts,
        for entity in feed.entity:
            if entity.HasField('vehicle'):
                vehicle = entity.vehicle
                position = vehicle.position
                
                data_list.append({
                    'ingestion_time': int(timestamp),
                    'id': entity.id,
                    'trip_id': vehicle.trip.trip_id,
                    'route_id': vehicle.trip.route_id,
                    'vehicle_id': vehicle.vehicle.id,
                    'latitude': position.latitude,
                    'longitude': position.longitude,
                    'bearing': position.bearing,
                    'current_status': vehicle.current_status
                })

        # 3. Create a temporary DataFrame and write it to the Bronze path
        # This simulates a continuous stream by writing small batches
        if data_list:
            df = spark.createDataFrame(data_list)
            
            # Use append mode to simulate continuous streaming data arrival
            df.write \
              .format("delta") \
              .mode("append") \
              .save(output_path)
            
            print(f"Fetched and appended {len(data_list)} vehicle positions.")
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
    except Exception as e:
        print(f"Error parsing data: {e}")


# demo run: to generate streaming data 

print("Starting simulated real-time ingestion...")
for _ in range(3): # Run a few times to simulate micro-batches
     fetch_and_save_realtime_data(API_URL, API_KEY, REALTIME_BRONZE_PATH)
     time.sleep(5) 
print("Simulated ingestion complete.")

In [0]:
# --- Define the Schema for the Bronze Stream ---
# We must define the schema strictly for streaming
bronze_schema = StructType([
    StructField("ingestion_time", LongType(), True),
    StructField("id", StringType(), True),
    StructField("trip_id", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("bearing", DoubleType(), True),
    StructField("current_status", IntegerType(), True),
])

# Define the stream from the Bronze Delta table
df_bronze_stream = (
    spark.readStream
    .format("delta")
    .load(REALTIME_BRONZE_PATH)
)

print(f"Defined Bronze Real-Time Stream from: {REALTIME_BRONZE_PATH}")
df_bronze_stream.printSchema()

In [0]:
# Silver Streaming Transformation: Cleaning and Standardizing

from pyspark.sql.functions import col, from_unixtime, current_timestamp, lit

REALTIME_SILVER_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/realtime_silver/vehicle_positions"
CHECKPOINT_LOCATION = f"{REALTIME_SILVER_PATH}/_checkpoint" # Required for Structured Streaming

# Clear stale checkpoint to avoid offset initialization errors on re-run
dbutils.fs.rm(CHECKPOINT_LOCATION, True)

print("\nStarting Silver Stream Transformation...")

df_silver_stream = (
    df_bronze_stream
    .select(
        # Convert UNIX time (Long) to Timestamp
        from_unixtime(col("ingestion_time")).cast("timestamp").alias("ingestion_timestamp"),
        
        # Standardize IDs as StringType
        col("trip_id").cast(StringType()).alias("trip_id"),
        col("route_id").cast(StringType()).alias("route_id"),
        col("vehicle_id").cast(StringType()).alias("vehicle_id"),
        
        # Location and Status Data
        col("latitude").cast(DoubleType()).alias("latitude"),
        col("longitude").cast(DoubleType()).alias("longitude"),
        col("bearing").cast(DoubleType()).alias("bearing"),
        
        
        col("current_status").alias("current_status_code"), 

        # Add a processing timestamp for auditing
        current_timestamp().alias("silver_processed_at")
    )
    # Filter out records missing critical identifiers (e.g., if trip_id is null)
    .filter(col("trip_id").isNotNull())
)

# Write to Silver Delta Table
(df_silver_stream.writeStream
    .format("delta")
    .outputMode("append") # Always append new positions
    .option("checkpointLocation", CHECKPOINT_LOCATION) # Required for recovery
    .option("path", REALTIME_SILVER_PATH)
    .trigger(processingTime='10 seconds') # Process micro-batches every 10 seconds (adjust as needed)
    .start()
)

print(f"Silver Real-Time Stream is RUNNING, writing to: {REALTIME_SILVER_PATH}")
print("Monitor the stream status in the Spark UI.")

In [0]:
df_silver_stream.display()

In [0]:
CATALOG_NAME = "bc_transit_ws"
SCHEMA_NAME = "silver"
TABLE_NAME = "realtime_vehicle_positions"

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")

full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

df_to_register = spark.read.format("delta").load(REALTIME_SILVER_PATH)

(df_to_register.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Registered Silver Realtime Table: {full_table_name}")